# Baseline NN for StripSearch Classification
Scope:
- preprocessing pipeline
- One-hot encoding for categorical features
- Standardization + PyTorch MLP baseline
- Core performance metrics and group-wise fairness snapshot


In [1]:
import random
import pickle

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 512
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 6

print("device:", DEVICE)

device: cpu


In [3]:
dataset = "./datasets/torontostripsearch.csv"
df = pd.read_csv(dataset)
print(df.columns)
df.head()

Index(['Arrest_Year', 'Arrest_Month', 'EventID', 'ArrestID', 'PersonID',
       'Perceived_Race', 'Sex', 'Age_group__at_arrest_',
       'Youth_at_arrest__under_18_years', 'ArrestLocDiv', 'StripSearch',
       'Booked', 'Occurrence_Category', 'Actions_at_arrest___Concealed_i',
       'Actions_at_arrest___Combative__', 'Actions_at_arrest___Resisted__d',
       'Actions_at_arrest___Mental_inst', 'Actions_at_arrest___Assaulted_o',
       'Actions_at_arrest___Cooperative', 'SearchReason_CauseInjury',
       'SearchReason_AssistEscape', 'SearchReason_PossessWeapons',
       'SearchReason_PossessEvidence', 'ItemsFound', 'ObjectId'],
      dtype='object')


,Arrest_Year,Arrest_Month,EventID,ArrestID,PersonID,Perceived_Race,Sex,Age_group__at_arrest_,Youth_at_arrest__under_18_years,ArrestLocDiv,...,Actions_at_arrest___Resisted__d,Actions_at_arrest___Mental_inst,Actions_at_arrest___Assaulted_o,Actions_at_arrest___Cooperative,SearchReason_CauseInjury,SearchReason_AssistEscape,SearchReason_PossessWeapons,SearchReason_PossessEvidence,ItemsFound,ObjectId
0,2020,July-Sept,1005907,6017884.0,326622,White,M,Aged 35 to 44 years,Not a youth,54,...,0,0,0,1,NaN,NaN,NaN,NaN,NaN,1
1,2020,July-Sept,1014562,6056669.0,326622,White,M,Aged 35 to 44 years,Not a youth,54,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,2
2,2020,Oct-Dec,1029922,6057065.0,326622,Unknown or Legacy,M,Aged 35 to 44 years,Not a youth,54,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,3
3,2021,Jan-Mar,1052190,6029059.0,327535,Black,M,Aged 25 to 34 years,Not a youth,XX,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,4
4,2021,Jan-Mar,1015512,6040372.0,327535,South Asian,M,Aged 25 to 34 years,Not a youth,XX,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,5


In [4]:
#missing data analysis
missing_data = df.isnull().sum()
print(missing_data[missing_data > 0])

ArrestID                          469
Perceived_Race                      4
Age_group__at_arrest_              24
Occurrence_Category               165
SearchReason_CauseInjury        57475
SearchReason_AssistEscape       57475
SearchReason_PossessWeapons     57475
SearchReason_PossessEvidence    57475
ItemsFound                      57475
dtype: int64


In [5]:
#identifiers and sparse columns analysis
for col in df.columns:
    unique_values = df[col].nunique()
    total_values = len(df[col])
    print(f"{col}: Unique={unique_values}, Total={total_values}, Missing={missing_data[col]}\n")

Arrest_Year: Unique=2, Total=65276, Missing=0

Arrest_Month: Unique=4, Total=65276, Missing=0

EventID: Unique=60003, Total=65276, Missing=0

ArrestID: Unique=64805, Total=65276, Missing=469

PersonID: Unique=37347, Total=65276, Missing=0

Perceived_Race: Unique=8, Total=65276, Missing=4

Sex: Unique=3, Total=65276, Missing=0

Age_group__at_arrest_: Unique=9, Total=65276, Missing=24

Youth_at_arrest__under_18_years: Unique=3, Total=65276, Missing=0

ArrestLocDiv: Unique=18, Total=65276, Missing=0

StripSearch: Unique=2, Total=65276, Missing=0

Booked: Unique=2, Total=65276, Missing=0

Occurrence_Category: Unique=31, Total=65276, Missing=165

Actions_at_arrest___Concealed_i: Unique=2, Total=65276, Missing=0

Actions_at_arrest___Combative__: Unique=2, Total=65276, Missing=0

Actions_at_arrest___Resisted__d: Unique=2, Total=65276, Missing=0

Actions_at_arrest___Mental_inst: Unique=2, Total=65276, Missing=0

Actions_at_arrest___Assaulted_o: Unique=2, Total=65276, Missing=0

Actions_at_arre

In [6]:
# Preprocessing steps 

# Dropping identifiers and very sparse columns that might not be useful for baseline learning.
fixed_drop_cols = [
    "ObjectId", "EventID", "ArrestID", "PersonID", "Booked", "ItemsFound",
    "SearchReason_CauseInjury", "SearchReason_AssistEscape",
    "SearchReason_PossessWeapons", "SearchReason_PossessEvidence"
]
existing_fixed_drop = [c for c in fixed_drop_cols if c in df.columns]
df = df.drop(columns=existing_fixed_drop)

#borrowed from Andrew's code to create a youth feature based on the "Youth_at_arrest__under_18_years" column. 
# remove rows with unknown sex label.
if "Sex" in df.columns:
    df = df[df["Sex"] != "U"].copy()

# This is a useful feature to have for the baseline model, and it also helps us understand how the youth status affects strip search decisions.
if "Youth_at_arrest__under_18_years" in df.columns:
    df["IsYouth"] = np.where(
        df["Youth_at_arrest__under_18_years"].fillna("Not a youth") == "Not a youth", 0, 1
    )

# Keep the quarter feature name consistent.
if "Arrest_Month" in df.columns:
    df = df.rename(columns={"Arrest_Month": "Arrest_Quarter"})

# Fill missing categoricals with explicit unknown tag.
for c in ["Perceived_Race", "Occurrence_Category", "ArrestLocDiv", "Age_group__at_arrest_", "Arrest_Quarter"]:
    if c in df.columns:
        df[c] = df[c].fillna("Unknown")

# Fill missing binary action columns with 0.
action_cols = [
    "Actions_at_arrest___Concealed_i",
    "Actions_at_arrest___Combative__",
    "Actions_at_arrest___Resisted__d",
    "Actions_at_arrest___Mental_inst",
    "Actions_at_arrest___Assaulted_o",
    "Actions_at_arrest___Cooperative",
]
for c in action_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

target_col = "StripSearch"
df = df.dropna(subset=[target_col]).copy()
df[target_col] = pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int)

feature_cols = [
    "Perceived_Race", "Sex", "Occurrence_Category", "ArrestLocDiv",
    "Age_group__at_arrest_", "Arrest_Quarter", "IsYouth",
    "Actions_at_arrest___Concealed_i", "Actions_at_arrest___Combative__",
    "Actions_at_arrest___Resisted__d", "Actions_at_arrest___Mental_inst",
    "Actions_at_arrest___Assaulted_o", "Actions_at_arrest___Cooperative",
]
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Rows:", len(df))
print("Features:", len(feature_cols))
print("Feature columns:", feature_cols)
print("Class distribution:")
print(y.value_counts(normalize=True).rename("ratio"))


Rows: 65267
Features: 13
Feature columns: ['Perceived_Race', 'Sex', 'Occurrence_Category', 'ArrestLocDiv', 'Age_group__at_arrest_', 'Arrest_Quarter', 'IsYouth', 'Actions_at_arrest___Concealed_i', 'Actions_at_arrest___Combative__', 'Actions_at_arrest___Resisted__d', 'Actions_at_arrest___Mental_inst', 'Actions_at_arrest___Assaulted_o', 'Actions_at_arrest___Cooperative']
Class distribution:
StripSearch
0    0.880476
1    0.119524
Name: ratio, dtype: float64


In [7]:
# Split the data into train and test sets with stratification to maintain class balance.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# simple step to identify categorical and numeric columns for later processing.
categorical_cols = [c for c in X_train_raw.columns if X_train_raw[c].dtype == "object"]
numeric_cols = [c for c in X_train_raw.columns if c not in categorical_cols]
print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

Categorical columns: ['Perceived_Race', 'Sex', 'Occurrence_Category', 'ArrestLocDiv', 'Age_group__at_arrest_', 'Arrest_Quarter']
Numeric columns: ['IsYouth', 'Actions_at_arrest___Concealed_i', 'Actions_at_arrest___Combative__', 'Actions_at_arrest___Resisted__d', 'Actions_at_arrest___Mental_inst', 'Actions_at_arrest___Assaulted_o', 'Actions_at_arrest___Cooperative']


In [8]:
#one hot encoding
def build_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", build_ohe(), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ],
    remainder="drop",
)

X_train_enc = preprocessor.fit_transform(X_train_raw)
X_test_enc = preprocessor.transform(X_test_raw)

scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_enc).astype(np.float32)
X_test_np = scaler.transform(X_test_enc).astype(np.float32)

y_train_np = y_train.to_numpy(dtype=np.float32)
y_test_np = y_test.to_numpy(dtype=np.float32)

print("X_train shape:", X_train_np.shape)
print("X_test shape:", X_test_np.shape)
print("Train positive ratio:", y_train_np.mean())

X_train shape: (52213, 82)
X_test shape: (13054, 82)
Train positive ratio: 0.11952962


In [9]:
#PyTorch datasets and loaders
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_np, y_train_np, test_size=0.1, random_state=SEED, stratify=y_train_np
)

train_ds = TensorDataset(
    torch.tensor(X_tr, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.float32),
)
val_ds = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In [10]:
# NN
class MLPBaseline(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

model = MLPBaseline(X_train_np.shape[1]).to(DEVICE)

pos = float(y_tr.sum())
neg = float(len(y_tr) - pos)
pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_state = None
best_val_loss = float("inf")
patience_ctr = 0


In [11]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_train_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * xb.size(0)

    train_loss = running_train_loss / len(train_ds)

    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            running_val_loss += loss.item() * xb.size(0)

    val_loss = running_val_loss / len(val_ds)
    print(f"epoch {epoch:02d} | train_loss={train_loss} | val_loss={val_loss}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print("early stopping triggered.")
            break

if best_state is not None:
    model.load_state_dict(best_state)


epoch 01 | train_loss=0.8858511618167426 | val_loss=0.7609351984343279
epoch 02 | train_loss=0.7281442653112774 | val_loss=0.7455460297472402
epoch 03 | train_loss=0.7081012842322397 | val_loss=0.737987914404071
epoch 04 | train_loss=0.6994112681686498 | val_loss=0.7465588236759102
epoch 05 | train_loss=0.691022299731603 | val_loss=0.7439965185695386
epoch 06 | train_loss=0.6831903130108428 | val_loss=0.7559727916357756
epoch 07 | train_loss=0.677920502063119 | val_loss=0.7586861054406957
epoch 08 | train_loss=0.6754423764155293 | val_loss=0.756099594022889
epoch 09 | train_loss=0.6677820635822159 | val_loss=0.7561181691906453
early stopping triggered.


In [12]:
model.eval()
with torch.no_grad():
    x_test_t = torch.tensor(X_test_np, dtype=torch.float32, device=DEVICE)
    test_logits = model(x_test_t)
    test_probs = torch.sigmoid(test_logits).cpu().numpy()

y_pred = (test_probs >= 0.5).astype(int)

acc = accuracy_score(y_test_np, y_pred)
f1 = f1_score(y_test_np, y_pred)
auc = roc_auc_score(y_test_np, test_probs)
cm = confusion_matrix(y_test_np, y_pred)

print(f"Accuracy: {acc}")
print(f"F1 Score: {f1}")
print(f"ROC-AUC: {auc}")
print("Confusion Matrix [rows=true, cols=pred]:")
print(cm)


Accuracy: 0.7922475869465297
F1 Score: 0.49421857515852297
ROC-AUC: 0.8964897237354607
Confusion Matrix [rows=true, cols=pred]:
[[9017 2477]
 [ 235 1325]]


In [13]:
# Group metrics function to evaluate performance across different groups defined by a specified column. 
# This will help us understand if there are disparities in model performance across different 
# demographic groups, which is crucial for fairness analysis.
def group_metrics(x_raw, y_true, y_hat, group_col):
    rows = []
    y_true = np.asarray(y_true).astype(int)
    y_hat = np.asarray(y_hat).astype(int)

    for group, idx in x_raw.groupby(group_col).groups.items():
        ids = np.array(list(idx))
        yt = y_true[x_raw.index.get_indexer(ids)]
        yp = y_hat[x_raw.index.get_indexer(ids)]

        tp = int(((yt == 1) & (yp == 1)).sum())
        tn = int(((yt == 0) & (yp == 0)).sum())
        fp = int(((yt == 0) & (yp == 1)).sum())
        fn = int(((yt == 1) & (yp == 0)).sum())

        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        selection_rate = (tp + fp) / max(len(yt), 1)

        rows.append({
            "group": group,
            "count": len(yt),
            "selection_rate": round(selection_rate, 4),
            "TPR": round(tpr, 4) if not np.isnan(tpr) else np.nan,
            "FPR": round(fpr, 4) if not np.isnan(fpr) else np.nan,
        })

    return pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)

for protected in ["Perceived_Race", "Sex"]:
    if protected in X_test_raw.columns:
        print(f"\nGroup metrics by {protected}:")
        display(group_metrics(X_test_raw, y_test_np, y_pred, protected))



Group metrics by Perceived_Race:


,group,count,selection_rate,TPR,FPR
0,White,5580,0.3201,0.8736,0.2391
1,Black,3483,0.3158,0.8719,0.2261
2,Unknown or Legacy,1030,0.2777,0.8067,0.2086
3,East/Southeast Asian,865,0.1896,0.7188,0.1473
4,South Asian,735,0.1660,0.7455,0.1191
5,Middle-Eastern,638,0.1803,0.6579,0.1500
6,Indigenous,393,0.3995,0.8906,0.3040
7,Latino,330,0.2182,0.6667,0.1830



Group metrics by Sex:


,group,count,selection_rate,TPR,FPR
0,M,10512,0.3054,0.8556,0.2281
1,F,2542,0.2329,0.8189,0.1647


In [14]:
# Saving the preprocessing artifacts and model state for future use. 
# artifact = {
#     "feature_cols": feature_cols,
#     "preprocessor": preprocessor,
#     "scaler": scaler,
#     "seed": SEED,
# }

# with open("baseline/nn_baseline_og_preprocess.pkl", "wb") as f:
#     pickle.dump(artifact, f)

# torch.save(model.state_dict(), "baseline/nn_baseline_og_model.pt")

# print("Saved:")
# print("- baseline/nn_baseline_og_preprocess.pkl")
# print("- baseline/nn_baseline_og_model.pt")
